In [55]:
# ============================================================
# CSCE-679 Data Visualization Assignment 1 — Level 1
# Hong Kong Year/Month Temperature Heatmap
# ============================================================

import json
from pathlib import Path

import pandas as pd
from IPython.display import HTML, display


# 1. Load and validate the daily temperature data

csv_path = Path("temperature_daily.csv")

if not csv_path.exists():
    from google.colab import files
    uploaded = files.upload()
    csv_path = Path("temperature_daily.csv")

if not csv_path.exists():
    raise FileNotFoundError(
        "Please upload the file named temperature_daily.csv."
    )

df = pd.read_csv(csv_path)
required_columns = {"date", "max_temperature", "min_temperature"}

if not required_columns.issubset(df.columns):
    raise ValueError(
        "CSV must contain: date, max_temperature, min_temperature"
    )

df["date"] = pd.to_datetime(df["date"], errors="coerce")
df["max_temperature"] = pd.to_numeric(
    df["max_temperature"], errors="coerce"
)
df["min_temperature"] = pd.to_numeric(
    df["min_temperature"], errors="coerce"
)

df = df.dropna(
    subset=["date", "max_temperature", "min_temperature"]
).copy()

df["year"] = df["date"].dt.year
df["month"] = df["date"].dt.month

# Aggregate daily records by year and month. The monthly maximum is the
# highest daily maximum, and the monthly minimum is the lowest daily minimum.
monthly = (
    df.groupby(["year", "month"], as_index=False)
      .agg(
          max_temperature=("max_temperature", "max"),
          min_temperature=("min_temperature", "min"),
      )
)

# Create a position for every month from 1997 through 2017. Missing months
# keep empty temperature values rather than being treated as zero.
FIRST_YEAR, LAST_YEAR = 1997, 2017
VIEWBOX_WIDTH, VIEWBOX_HEIGHT = 1080, 475
years = list(range(FIRST_YEAR, LAST_YEAR + 1))

lookup = {
    (int(row.year), int(row.month)): {
        "max_temperature": float(row.max_temperature),
        "min_temperature": float(row.min_temperature),
    }
    for row in monthly.itertuples(index=False)
}

matrix_data = []

for year in years:
    for month in range(1, 13):
        record = lookup.get((year, month))
        matrix_data.append({
            "year": year,
            "month": month,
            "max_temperature": (
                record["max_temperature"] if record else None
            ),
            "min_temperature": (
                record["min_temperature"] if record else None
            ),
        })

data_json = json.dumps(matrix_data)

# Print selected monthly summaries to verify the aggregation.
for year, month in [(2002, 3), (2007, 11), (2011, 1), (2017, 8)]:
    record = lookup.get((year, month))
    print(f"{year}-{month:02d}: {record}")


# 2. Create the interactive heatmap

html = r"""
<div id="level1-container" style="
    font-family: Arial, sans-serif;
    color: #111;
    width: 100%;
">
  <h2 style="font-size:24px; margin:0 0 8px 0;">
    Level 1: Year/Month Heatmap
  </h2>
  <h3 style="font-size:20px;font-weight:500;margin:0 0 12px 0;">HK Temperature Heatmap</h3>

  <label for="temperature-mode" style="font-size:18px;">
    Temperature:
  </label>

  <select id="temperature-mode" style="
      font-size:16px;
      padding:4px 6px;
      margin-left:4px;
      margin-bottom:8px;
  ">
    <option value="max_temperature">Maximum temperature</option>
    <option value="min_temperature">Minimum temperature</option>
  </select>

  <div id="heatmap-holder" style="width:100%; overflow-x:auto;">
    <svg id="level1-svg"
         viewBox="0 0 __VIEWBOX_WIDTH__ __VIEWBOX_HEIGHT__"
         style="width:__VIEWBOX_WIDTH__px; max-width:none; height:auto; display:block;">
<g id="heatmap-chart"></g>
    </svg>
  </div>

  <div id="level1-tooltip" style="
      display:none;
      position:fixed;
      z-index:99999;
      background:#fff;
      border:1px solid #888;
      padding:7px 9px;
      border-radius:3px;
      font-size:13px;
      line-height:1.45;
      pointer-events:none;
      box-shadow:0 1px 4px #0002;
  "></div>
</div>

<script src="https://d3js.org/d3.v7.min.js"></script>
<script>
(function () {
  const data = __DATA_JSON__;

  // Keep the chart dimensions and spacing together so the layout is easy to adjust.
  const CONFIG = {
    firstYear: __FIRST_YEAR__,
    lastYear: __LAST_YEAR__,
    cellWidth: 31,
    cellHeight: 31,
    xStart: 60,
    yStart: 28,
    xGap: 9.5,
    yGap: 5.7,
    legendX: 960,
    legendWidth: 18,
    temperatureMin: 0,
    temperatureMax: 40,
    yearFontSize: 9,
    monthFontSize: 10,
    axisOffset: 6,
    tickLength: 4,
    labelOffset: 5,
    axisStrokeWidth: 0.8,
    cellStrokeWidth: 1,
    legendLabelFontSize: 9,
    yearLabelOffset: 9,
    monthLabelBaselineOffset: 3,
    legendLabelOffset: 7,
    tooltipOffset: 12,
    tooltipEdgePadding: 8
  };

  const years = d3.range(CONFIG.firstYear, CONFIG.lastYear + 1);
  const months = [
    "January", "February", "March", "April",
    "May", "June", "July", "August",
    "September", "October", "November", "December"
  ];

  const chart = d3.select("#heatmap-chart");
  const tooltip = document.getElementById("level1-tooltip");
  const select = document.getElementById("temperature-mode");

  function positionTooltip(event) {
    const left = Math.min(
      event.clientX + CONFIG.tooltipOffset,
      window.innerWidth - tooltip.offsetWidth - CONFIG.tooltipEdgePadding
    );
    const top = Math.min(
      event.clientY + CONFIG.tooltipOffset,
      window.innerHeight - tooltip.offsetHeight - CONFIG.tooltipEdgePadding
    );
    tooltip.style.left = `${Math.max(CONFIG.tooltipEdgePadding, left)}px`;
    tooltip.style.top = `${Math.max(CONFIG.tooltipEdgePadding, top)}px`;
  }

  // Band scales place each year and month in its own evenly spaced position.
  const xScale = d3.scaleBand()
    .domain(years)
    .range([CONFIG.xStart, CONFIG.xStart + years.length * CONFIG.cellWidth + (years.length - 1) * CONFIG.xGap])
    .paddingInner(CONFIG.xGap / (CONFIG.cellWidth + CONFIG.xGap))
    .paddingOuter(0);
  const yScale = d3.scaleBand()
    .domain(months.map((_, index) => index + 1))
    .range([CONFIG.yStart, CONFIG.yStart + months.length * CONFIG.cellHeight + (months.length - 1) * CONFIG.yGap])
    .paddingInner(CONFIG.yGap / (CONFIG.cellHeight + CONFIG.yGap))
    .paddingOuter(0);

  const cellWidth = xScale.bandwidth();
  const cellHeight = yScale.bandwidth();
  const xPosition = year => xScale(year);
  const yPosition = month => yScale(month);
  const xCenter = year => xScale(year) + xScale.bandwidth() / 2;
  const yCenter = month => yScale(month) + yScale.bandwidth() / 2;

  // Use one fixed temperature scale for both maximum and minimum values.
  // This keeps the same temperature mapped to the same color in both levels.
  const spectralColors = d3.schemeSpectral[11].slice().reverse();
  const temperatureThresholds = [4, 8, 12, 15, 18, 22, 25, 29, 33, 36];
  const temperatureColor = d3.scaleThreshold()
    .domain(temperatureThresholds)
    .range(spectralColors);

  const cells = chart.selectAll(".temperature-cell")
    .data(data, d => `${d.year}-${d.month}`)
    .join("rect")
    .attr("class", "temperature-cell")
    .attr("x", d => xPosition(d.year))
    .attr("y", d => yPosition(d.month))
    .attr("width", cellWidth)
    .attr("height", cellHeight)
    .attr("stroke", "#ffffff")
    .attr("stroke-width", CONFIG.cellStrokeWidth)
    .style("cursor", d =>
      d.max_temperature === null ? "default" : "pointer"
    )
    .on("mouseenter", function(event, d) {
      if (d.max_temperature === null) return;

      tooltip.style.display = "block";
      tooltip.innerHTML =
        `<strong>Date:</strong> ${d.year}-${String(d.month).padStart(2, "0")}; ` +
        `<strong>max:</strong> ${d.max_temperature}°C ` +
        `<strong>min:</strong> ${d.min_temperature}°C`;

      positionTooltip(event);
    })
    .on("mousemove", function(event) {
      positionTooltip(event);
    })
    .on("mouseleave", function() {
      tooltip.style.display = "none";
    });

  // Draw the top and left axes with ticks and labels.
  const gridTop = CONFIG.yStart - CONFIG.axisOffset;
  const gridBottom = yScale(months.length) + yScale.bandwidth();
  const gridLeft = CONFIG.xStart - CONFIG.axisOffset;
  const gridRight = xScale(CONFIG.lastYear) + xScale.bandwidth();

  chart.append("line")
    .attr("x1", CONFIG.xStart).attr("x2", gridRight)
    .attr("y1", gridTop).attr("y2", gridTop)
    .attr("stroke", "#333").attr("stroke-width", CONFIG.axisStrokeWidth);

  chart.selectAll(".year-tick")
    .data(years)
    .join("line")
    .attr("class", "year-tick")
    .attr("x1", year => xCenter(year)).attr("x2", year => xCenter(year))
    .attr("y1", gridTop - CONFIG.tickLength).attr("y2", gridTop)
    .attr("stroke", "#333").attr("stroke-width", CONFIG.axisStrokeWidth);

  chart.selectAll(".year-label")
    .data(years)
    .join("text")
    .attr("class", "year-label")
    .attr("x", year => xCenter(year))
    .attr("y", gridTop - CONFIG.yearLabelOffset)
    .attr("text-anchor", "middle")
    .attr("font-size", CONFIG.yearFontSize)
    .text(year => year);

  // Draw the left axis line, month tick marks, and month labels.
  chart.append("line")
    .attr("x1", gridLeft).attr("x2", gridLeft)
    .attr("y1", CONFIG.yStart).attr("y2", gridBottom)
    .attr("stroke", "#333").attr("stroke-width", CONFIG.axisStrokeWidth);

  chart.selectAll(".month-tick")
    .data(months)
    .join("line")
    .attr("class", "month-tick")
    .attr("x1", gridLeft - CONFIG.tickLength).attr("x2", gridLeft)
    .attr("y1", (_, index) => yCenter(index + 1))
    .attr("y2", (_, index) => yCenter(index + 1))
    .attr("stroke", "#333").attr("stroke-width", CONFIG.axisStrokeWidth);

  chart.selectAll(".month-label")
    .data(months)
    .join("text")
    .attr("class", "month-label")
    .attr("x", gridLeft - CONFIG.labelOffset)
    .attr("y", (_, index) => yCenter(index + 1) + CONFIG.monthLabelBaselineOffset)
    .attr("text-anchor", "end")
    .attr("font-size", CONFIG.monthFontSize)
    .text(month => month);

  // Draw discrete legend bands using the same colors as the heatmap.
  const legendX = CONFIG.legendX;
  const legendY = CONFIG.yStart;
  const legendWidth = CONFIG.legendWidth;
  const legendHeight = gridBottom - gridTop;
  const bandHeight = legendHeight / spectralColors.length;

  chart.selectAll(".legend-band")
    .data(spectralColors)
    .join("rect")
    .attr("class", "legend-band")
    .attr("x", legendX)
    .attr("y", (_, i) => legendY + i * bandHeight)
    .attr("width", legendWidth)
    .attr("height", bandHeight + 0.2)
    .attr("fill", color => color);

  // Label the endpoints and each temperature threshold used by the color scale.
  const legendLabels = [CONFIG.temperatureMin, ...temperatureThresholds, CONFIG.temperatureMax];
  chart.selectAll(".legend-label")
    .data(legendLabels)
    .join("text")
    .attr("class", "legend-label")
    .attr("x", legendX + legendWidth + CONFIG.legendLabelOffset)
    .attr("y", (_, index) => legendY + index * bandHeight)
    .attr("font-size", CONFIG.legendLabelFontSize)
    .attr("dominant-baseline", "middle")
    .text(value => `${value} Celsius`);

  function updateColors() {
    const metric = select.value;

    cells.attr("fill", d => {
      const value = d[metric];
      if (value === null || !Number.isFinite(value)) {
        return "#eeeeee";
      }

      return temperatureColor(value);
    });
  }

  select.addEventListener("change", updateColors);
  updateColors();
})();
</script>
"""

html = (html.replace("__DATA_JSON__", data_json)
        .replace("__FIRST_YEAR__", str(FIRST_YEAR))
        .replace("__LAST_YEAR__", str(LAST_YEAR))
        .replace("__VIEWBOX_WIDTH__", str(VIEWBOX_WIDTH))
        .replace("__VIEWBOX_HEIGHT__", str(VIEWBOX_HEIGHT)))
display(HTML(html))

2002-03: {'max_temperature': 28.0, 'min_temperature': 14.0}
2007-11: {'max_temperature': 29.0, 'min_temperature': 12.0}
2011-01: {'max_temperature': 18.0, 'min_temperature': 6.0}
2017-08: {'max_temperature': 37.0, 'min_temperature': 25.0}


In [56]:
# ============================================================
# CSCE-679 Data Visualization Assignment 1 — Level 2
# Year/Month Heatmap with Daily Temperature Mini-Charts
# ============================================================

import json
from pathlib import Path

import pandas as pd
from IPython.display import HTML, display

# 1. Load and validate the daily temperature data

# Find the CSV file containing the daily temperature records.
CSV_PATH = Path("temperature_daily.csv")

if not CSV_PATH.exists():
    from google.colab import files
    files.upload()

if not CSV_PATH.exists():
    raise FileNotFoundError(
        "Please upload temperature_daily.csv and run this cell again."
    )

# Read the CSV file into a DataFrame so that the data can be processed.
df = pd.read_csv(CSV_PATH)

required_columns = {
    "date", "max_temperature", "min_temperature"
}

if not required_columns.issubset(df.columns):
    raise ValueError(
        "CSV must contain date, max_temperature, and min_temperature."
    )

# Convert the date and temperature columns to usable data types.
df["date"] = pd.to_datetime(df["date"], errors="coerce")
df["max_temperature"] = pd.to_numeric(
    df["max_temperature"], errors="coerce"
)
df["min_temperature"] = pd.to_numeric(
    df["min_temperature"], errors="coerce"
)

df = df.dropna(
    subset=["date", "max_temperature", "min_temperature"]
).copy()

df["year"] = df["date"].dt.year
df["month"] = df["date"].dt.month
df["day"] = df["date"].dt.day

# Define the displayed year range once for both Python and JavaScript.
FIRST_YEAR, LAST_YEAR = 2008, 2017
VIEWBOX_WIDTH, VIEWBOX_HEIGHT = 1146, 936
df = df[df["year"].between(FIRST_YEAR, LAST_YEAR)].sort_values("date")

# 2. Prepare the monthly summaries and daily records
# Aggregate daily records by month: maximum of daily maxima and minimum of daily minima.
monthly = (
    df.groupby(["year", "month"], as_index=False)
      .agg(
          max_temperature=("max_temperature", "max"),
          min_temperature=("min_temperature", "min"),
      )
)

# Prepare each daily observation for use in the JavaScript visualization.
daily_data = [
    {
        "date": r.date.strftime("%Y-%m-%d"),
        "year": int(r.year),
        "month": int(r.month),
        "day": int(r.day),
        "max_temperature": float(r.max_temperature),
        "min_temperature": float(r.min_temperature),
    }
    for r in df.itertuples(index=False)
]

# Prepare the monthly summaries for the heatmap and tooltips.
monthly_data = [
    {
        "year": int(r.year),
        "month": int(r.month),
        "max_temperature": float(r.max_temperature),
        "min_temperature": float(r.min_temperature),
    }
    for r in monthly.itertuples(index=False)
]

print("Daily observations:", len(daily_data))
print("Monthly cells with data:", len(monthly_data))

# Printing January 2011 temperatures to check the monthly summary.
jan2011 = monthly[
    (monthly["year"] == 2011) & (monthly["month"] == 1)
]

if not jan2011.empty:
    print("January 2011 monthly maximum:",
          jan2011.iloc[0]["max_temperature"], "°C")
    print("January 2011 monthly minimum:",
          jan2011.iloc[0]["min_temperature"], "°C")

# 3. Create the interactive visualization

html = r"""
<div id="level2-container"
     style="font-family:Arial,sans-serif;color:#111;width:100%;">
  <h2 style="font-size:24px; margin:0 0 8px 0;">Level 2: Daily Temperature Heatmap</h2>
  <h3 style="font-size:20px;font-weight:500;margin:0 0 12px 0;">HK Temperature Heatmap</h3>

  <div style="width:100%;overflow-x:auto;">
    <svg id="level2-svg"
         viewBox="0 0 __VIEWBOX_WIDTH__ __VIEWBOX_HEIGHT__"
         style="width:__VIEWBOX_WIDTH__px;max-width:none;height:auto;display:block;">
      <g id="level2-chart"></g>
    </svg>
  </div>

  <div id="level2-tooltip" style="
      display:none;position:fixed;z-index:99999;
      background:white;border:1px solid #888;
      padding:7px 9px;border-radius:3px;
      font:12px/1.5 Arial;pointer-events:none;
      box-shadow:0 1px 4px #0002;">
  </div>
</div>

<script src="https://d3js.org/d3.v7.min.js"></script>
<script>
(function () {
  // Loading the daily records and monthly summaries passed from Python.
  const daily = __DAILY_JSON__;
  const monthly = __MONTHLY_JSON__;

  // Keep the layout settings together so chart spacing can be changed easily.
  const CONFIG = {
    firstYear: __FIRST_YEAR__,
    lastYear: __LAST_YEAR__,
    cellWidth: 78,
    cellHeight: 58,
    xStart: 88,
    yStart: 40,
    xGap: 19,
    yGap: 14,
    legendX: 1059,
    legendY: 48,
    legendWidth: 22,
    legendHeight: 242,
    temperatureMin: 0,
    temperatureMax: 40,
    yearFontSize: 12,
    monthFontSize: 11,
    dailyLineMaxColor: "#00b85a",
    dailyLineMinColor: "#43cfe3",
    axisOffset: 6,
    tickLength: 4,
    labelOffset: 5,
    yearLabelOffset: 12,
    monthLabelBaselineOffset: 4,
    axisStrokeWidth: 0.8,
    cellStrokeWidth: 1,
    dailyLineStrokeWidth: 1.5,
    dayMin: 1,
    dayMax: 31,
    dayInnerPadding: 2,
    temperatureInnerPadding: 3,
    legendLabelFontSize: 10,
    legendLabelOffset: 6,
    tooltipOffset: 12,
    tooltipEdgePadding: 8
  };

  const years = d3.range(CONFIG.firstYear, CONFIG.lastYear + 1);
  const months = [
    "January", "February", "March", "April",
    "May", "June", "July", "August",
    "September", "October", "November", "December"
  ];

  const chart = d3.select("#level2-chart");
  const tooltip = document.getElementById("level2-tooltip");

  function positionTooltip(event) {
    const left = Math.min(
      event.clientX + CONFIG.tooltipOffset,
      window.innerWidth - tooltip.offsetWidth - CONFIG.tooltipEdgePadding
    );
    const top = Math.min(
      event.clientY + CONFIG.tooltipOffset,
      window.innerHeight - tooltip.offsetHeight - CONFIG.tooltipEdgePadding
    );
    tooltip.style.left = `${Math.max(CONFIG.tooltipEdgePadding, left)}px`;
    tooltip.style.top = `${Math.max(CONFIG.tooltipEdgePadding, top)}px`;
  }

  // Band scales position the years and months while preserving gaps between cells.
  const xScale = d3.scaleBand()
    .domain(years)
    .range([CONFIG.xStart, CONFIG.xStart + years.length * CONFIG.cellWidth + (years.length - 1) * CONFIG.xGap])
    .paddingInner(CONFIG.xGap / (CONFIG.cellWidth + CONFIG.xGap))
    .paddingOuter(0);
  const yScale = d3.scaleBand()
    .domain(months.map((_, index) => index + 1))
    .range([CONFIG.yStart, CONFIG.yStart + months.length * CONFIG.cellHeight + (months.length - 1) * CONFIG.yGap])
    .paddingInner(CONFIG.yGap / (CONFIG.cellHeight + CONFIG.yGap))
    .paddingOuter(0);

  const cellWidth = xScale.bandwidth();
  const cellHeight = yScale.bandwidth();
  const xPosition = year => xScale(year);
  const yPosition = month => yScale(month);
  const xCenter = year => xScale(year) + xScale.bandwidth() / 2;
  const yCenter = month => yScale(month) + yScale.bandwidth() / 2;

  // Use the same reversed Spectral palette and temperature thresholds as Level 1.
  const spectralColors = d3.schemeSpectral[11].slice().reverse();

  const temperatureThresholds = [4, 8, 12, 15, 18, 22, 25, 29, 33, 36];
  const temperatureColor = d3.scaleThreshold()
    .domain(temperatureThresholds)
    .range(spectralColors);

  // Organizing the monthly summaries and daily records for each cell.

  // Finding a monthly summary quickly using its year and month.
  const monthlyByKey = new Map(
    monthly.map(d => [`${d.year}-${d.month}`, d])
  );

  // Group the daily records by year and month so each cell gets its daily values.
  const dailyByKey = d3.group(
    daily,
    d => `${d.year}-${d.month}`
  );

  // Building every year/month cell, including months without available data.
  const cells = [];

  for (const year of years) {
    for (let month = 1; month <= 12; month++) {
      const key = `${year}-${month}`;

      cells.push({
        year,
        month,
        summary: monthlyByKey.get(key) || null,
        days: (dailyByKey.get(key) || [])
          .slice()
          .sort((a, b) => a.day - b.day)
      });
    }
  }

  // Draw each monthly background using the monthly maximum temperature.

  chart.selectAll(".month-background")
    .data(cells)
    .join("rect")
    .attr("class", "month-background")
    .attr("x", d => xPosition(d.year))
    .attr("y", d => yPosition(d.month))
    .attr("width", cellWidth)
    .attr("height", cellHeight)
    // Use the monthly maximum for the background color so each cell has
    // one summary color, while the two lines show daily max and min changes.
    .attr("fill", d =>
      d.summary
        ? temperatureColor(d.summary.max_temperature)
        : "#eeeeee"
    )
    .attr("stroke", "#ffffff")
    .attr("stroke-width", CONFIG.cellStrokeWidth);

  // Draw the top axis line, year tick marks, and year labels.
  const gridTop = CONFIG.yStart - CONFIG.axisOffset;
  const gridBottom = yScale(months.length) + yScale.bandwidth();
  const gridLeft = CONFIG.xStart - CONFIG.axisOffset;
  const gridRight = xScale(CONFIG.lastYear) + xScale.bandwidth();

  chart.append("line")
    .attr("x1", CONFIG.xStart).attr("x2", gridRight)
    .attr("y1", gridTop).attr("y2", gridTop)
    .attr("stroke", "#333").attr("stroke-width", CONFIG.axisStrokeWidth);

  chart.selectAll(".year-tick")
    .data(years)
    .join("line")
    .attr("class", "year-tick")
    .attr("x1", year => xCenter(year)).attr("x2", year => xCenter(year))
    .attr("y1", gridTop - CONFIG.tickLength).attr("y2", gridTop)
    .attr("stroke", "#333").attr("stroke-width", CONFIG.axisStrokeWidth);

  chart.selectAll(".year-label")
    .data(years)
    .join("text")
    .attr("class", "year-label")
    .attr("x", year => xCenter(year))
    .attr("y", gridTop - CONFIG.yearLabelOffset)
    .attr("text-anchor", "middle")
    .attr("font-size", CONFIG.yearFontSize)
    .text(year => year);

  // Draw the left axis line, month tick marks, and month labels.
  chart.append("line")
    .attr("x1", gridLeft).attr("x2", gridLeft)
    .attr("y1", CONFIG.yStart).attr("y2", gridBottom)
    .attr("stroke", "#333").attr("stroke-width", CONFIG.axisStrokeWidth);

  chart.selectAll(".month-tick")
    .data(months)
    .join("line")
    .attr("class", "month-tick")
    .attr("x1", gridLeft - CONFIG.tickLength).attr("x2", gridLeft)
    .attr("y1", (_, index) => yCenter(index + 1))
    .attr("y2", (_, index) => yCenter(index + 1))
    .attr("stroke", "#333").attr("stroke-width", CONFIG.axisStrokeWidth);

  chart.selectAll(".month-label")
    .data(months)
    .join("text")
    .attr("class", "month-label")
    .attr("x", gridLeft - CONFIG.labelOffset)
    .attr("y", (_, index) => yCenter(index + 1) + CONFIG.monthLabelBaselineOffset)
    .attr("text-anchor", "end")
    .attr("font-size", CONFIG.monthFontSize)
    .text(month => month);

  // Setting the scales used to position days and temperatures in each cell.

  // Map day numbers to horizontal positions inside each cell.
  const xDay = d3.scaleLinear()
    .domain([CONFIG.dayMin, CONFIG.dayMax])
    .range([CONFIG.dayInnerPadding, cellWidth - CONFIG.dayInnerPadding]);

  // Mapping temperature values to vertical positions inside each cell.
  const yTemperature = d3.scaleLinear()
    .domain([CONFIG.temperatureMin, CONFIG.temperatureMax])
    .range([cellHeight - CONFIG.temperatureInnerPadding, CONFIG.temperatureInnerPadding])
    .clamp(true);

  // Use one line generator for either daily temperature field.
  function makeLineGenerator(field) {
    return d3.line()
      .defined(d => Number.isFinite(d[field]))
      .x(d => xDay(d.day))
      .y(d => yTemperature(d[field]));
  }

  const maximumLine = makeLineGenerator("max_temperature");
  const minimumLine = makeLineGenerator("min_temperature");

  // Drawing the daily maximum and minimum temperature lines in each month.

  const cellsWithData = cells.filter(d => d.days.length > 0);

  const lineGroups = chart.selectAll(".daily-lines")
    .data(cellsWithData)
    .join("g")
    .attr("class", "daily-lines")
    .attr("transform", d =>
      `translate(${xPosition(d.year)},${yPosition(d.month)})`
    );

  // Drawing the daily maximum temperatures in green.
  lineGroups.append("path")
    .attr("class", "maximum-line")
    .attr("d", d => maximumLine(d.days))
    .attr("fill", "none")
    .attr("stroke", CONFIG.dailyLineMaxColor)
    .attr("stroke-width", CONFIG.dailyLineStrokeWidth)
    .attr("stroke-linejoin", "round")
    .attr("stroke-linecap", "round")
    .style("pointer-events", "none");

  // Drawing the daily minimum temperatures in cyan.
  lineGroups.append("path")
    .attr("class", "minimum-line")
    .attr("d", d => minimumLine(d.days))
    .attr("fill", "none")
    .attr("stroke", CONFIG.dailyLineMinColor)
    .attr("stroke-width", CONFIG.dailyLineStrokeWidth)
    .attr("stroke-linejoin", "round")
    .attr("stroke-linecap", "round")
    .style("pointer-events", "none");

  // Draw the discrete legend using the same thresholds as the heatmap.
  const legendX = CONFIG.legendX;
  const legendY = CONFIG.legendY;
  const legendWidth = CONFIG.legendWidth;
  const legendHeight = CONFIG.legendHeight;
  const bandHeight = legendHeight / spectralColors.length;

  chart.selectAll(".legend-band")
    .data(spectralColors)
    .join("rect")
    .attr("class", "legend-band")
    .attr("x", legendX)
    .attr("y", (_, i) => legendY + i * bandHeight)
    .attr("width", legendWidth)
    .attr("height", bandHeight + 0.2)
    .attr("fill", color => color);

  // Label the endpoints and each temperature threshold used by the color scale.
  const legendLabels = [CONFIG.temperatureMin, ...temperatureThresholds, CONFIG.temperatureMax];
  chart.selectAll(".legend-label")
    .data(legendLabels)
    .join("text")
    .attr("class", "legend-label")
    .attr("x", legendX + legendWidth + CONFIG.legendLabelOffset)
    .attr("y", (_, index) => legendY + index * bandHeight)
    .attr("font-size", CONFIG.legendLabelFontSize)
    .attr("dominant-baseline", "middle")
    .text(value => `${value} Celsius`);

  // Showing the monthly temperature summary when hovering over a cell.

  // Adding a transparent layer over each cell to detect mouse movement.
  chart.selectAll(".cell-overlay")
    .data(cells)
    .join("rect")
    .attr("class", "cell-overlay")
    .attr("x", d => xPosition(d.year))
    .attr("y", d => yPosition(d.month))
    .attr("width", cellWidth)
    .attr("height", cellHeight)
    .attr("fill", "transparent")
    .style("cursor", d => d.summary ? "pointer" : "default")
    .on("mouseenter", (event, d) => {
      if (!d.summary) return;

      tooltip.style.display = "block";
      tooltip.innerHTML =
        `<strong>Date:</strong> ${d.year}-${String(d.month).padStart(2, "0")}; ` +
        `<strong>max:</strong> ${d.summary.max_temperature}°C ` +
        `<strong>min:</strong> ${d.summary.min_temperature}°C`;

      positionTooltip(event);
    })
    .on("mousemove", event => {
      positionTooltip(event);
    })
    .on("mouseleave", () => {
      tooltip.style.display = "none";
    });
})();
</script>
"""

# Insert the processed data into the HTML and display the visualization.
html = (
    html.replace("__DAILY_JSON__", json.dumps(daily_data))
        .replace("__MONTHLY_JSON__", json.dumps(monthly_data))
        .replace("__FIRST_YEAR__", str(FIRST_YEAR))
        .replace("__LAST_YEAR__", str(LAST_YEAR))
        .replace("__VIEWBOX_WIDTH__", str(VIEWBOX_WIDTH))
        .replace("__VIEWBOX_HEIGHT__", str(VIEWBOX_HEIGHT))
)

display(HTML(html))

Daily observations: 3587
Monthly cells with data: 118
January 2011 monthly maximum: 18 °C
January 2011 monthly minimum: 6 °C
